# WarpLab Notebook

This notebook is the notebook-first interface for WarpLab.

It follows the intended experimentation loop:

1. environment setup and runtime checks
2. CUDA kernel inspection
3. compile step
4. benchmark runner
5. optimization loop
6. performance visualization


## 1. Environment Setup And Runtime Checks

If you are in Colab or Kaggle, make sure the repo is present in the runtime and the runtime actually exposes a GPU with CUDA compiler access.


In [ ]:
from pathlib import Path

ROOT_DIR = Path('.').resolve()
if not (ROOT_DIR / 'pyproject.toml').exists():
    ROOT_DIR = ROOT_DIR.parent.resolve()

PROJECT_PATH = ROOT_DIR / 'projects' / 'saxpy'
ROOT_DIR, PROJECT_PATH


In [ ]:
from warplab.cloud import collect_runtime_diagnostics, notebook_bootstrap_snippet, runtime_warnings

diagnostics = collect_runtime_diagnostics()
warnings = runtime_warnings(diagnostics)

print('Bootstrap snippet for cloud notebooks:')
print(notebook_bootstrap_snippet())
print('\nRuntime diagnostics:')
diagnostics


In [ ]:
if warnings:
    print('Runtime warnings:')
    for warning in warnings:
        print('-', warning)
else:
    print('Runtime looks ready for CUDA compilation and profiling.')


## 2. CUDA Kernel Inspection

Inspect the current kernel source here. Edit `projects/saxpy/kernel.cu` in the file browser or your editor, then rerun the compile and benchmark cells.


In [ ]:
kernel_path = PROJECT_PATH / 'kernel.cu'
print(kernel_path.read_text())


## 3. Compile Step

Compile the baseline benchmark and validator binaries explicitly.


In [ ]:
from warplab.compiler import compile_kernel
from warplab.config import load_project_config

config = load_project_config(PROJECT_PATH)
RUN_DIR = ROOT_DIR / 'runs' / 'notebook_manual'
RUN_DIR.mkdir(parents=True, exist_ok=True)

baseline_bench = RUN_DIR / 'baseline_bench'
baseline_validate = RUN_DIR / 'baseline_validate'

bench_compile = compile_kernel(PROJECT_PATH, config.build['compile_kernel'], baseline_bench)
validate_compile = compile_kernel(PROJECT_PATH, config.build['compile_validate'], baseline_validate)

print('bench compile ok:', bench_compile.success)
print('validate compile ok:', validate_compile.success)
print(bench_compile.stderr)
print(validate_compile.stderr)


## 4. Benchmark Runner

Validate the baseline, then benchmark it with explicit warmup and timed-run counts.


In [ ]:
from warplab.validator import run_validator
from warplab.benchmark import run_benchmark

run_context = {**config.input, **config.validation}

validation = run_validator(baseline_validate, config.run['validate'], cwd=PROJECT_PATH, extra_context=run_context)
benchmark = run_benchmark(
    baseline_bench,
    config.run['benchmark'],
    warmup_runs=config.budget['warmup_runs'],
    timed_runs=config.budget['timed_runs'],
    cwd=PROJECT_PATH,
    extra_context=run_context,
)

print('validation valid:', validation.valid)
print('baseline median ms:', benchmark.median_ms)
print('baseline cv:', benchmark.cv)


## 5. Optimization Loop

Run the full WarpLab orchestration layer from the notebook.


In [ ]:
from warplab.runner import run_project

result = run_project(PROJECT_PATH, ROOT_DIR, candidate_count=12)
result['run_id'], result['report_path'], result['run_summary_path']


## 6. Performance Visualization

Inspect the top candidates and visualize the search results.


In [ ]:
from warplab.visualize import (
    plot_parameter_impact,
    plot_performance_distribution,
    plot_stability_vs_speedup,
    plot_top_candidates,
    prepare_results_dataframe,
)

df = prepare_results_dataframe(result['results'])
display(df[['id', 'latency_ms', 'speedup', 'cv', 'score', 'flags']].head(10))

plot_performance_distribution(df)
plot_stability_vs_speedup(df)
plot_top_candidates(df)
if config.search_space:
    first_param = list(config.search_space.keys())[0]
    plot_parameter_impact(df, first_param)
